# 02_collate_tactical_dataset.ipynb — Minimap Crop and Match Collation

This notebook contains the logic to scan raw gameplay screenshots from the project directories, crop the bottom-center circular minimap (radar overlay), preprocess the main frame view, and generate a JSON mapping metadata file used to train the `TacticalVisionNet` model.

In [ ]:
import os
import json
import random
from pathlib import Path
from PIL import Image

def crop_minimap(img):
    w, h = img.size
    # Standard circular minimap positioning: bottom center
    center_x = w // 2
    center_y = int(h * 0.835)
    radius = int(h * 0.125)
    
    left = max(0, center_x - radius)
    top = max(0, center_y - radius)
    right = min(w, center_x + radius)
    bottom = min(h, center_y + radius)
    
    cropped = img.crop((left, top, right, bottom))
    try:
        resample = Image.Resampling.LANCZOS
    except AttributeError:
        resample = Image.LANCZOS
    return cropped.resize((224, 224), resample)

print("Crop function defined successfully.")

In [ ]:
# Configure source directories and output targets
input_directories = ["../static_screenshots", "../screenshots", "../screenshots/static"]
output_dir = Path("../artifacts/collate_cache")
output_dir.mkdir(parents=True, exist_ok=True)

crops_path = output_dir / "minimap_crops"
crops_path.mkdir(parents=True, exist_ok=True)

records = []
extensions = ("*.jpg", "*.jpeg", "*.png")
found_files = []

for d in input_directories:
    path = Path(d)
    if not path.exists():
        continue
    for ext in extensions:
        found_files.extend(path.glob(ext))
        
print(f"Found {len(found_files)} screenshots for processing.")

for idx, filepath in enumerate(found_files):
    try:
        img = Image.open(filepath).convert("RGB")
        
        # 1. Main frame preprocessing (resize to 224x224)
        try:
            resample = Image.Resampling.LANCZOS
        except AttributeError:
            resample = Image.LANCZOS
        main_resized = img.resize((224, 224), resample)
        main_filename = f"main_{idx}.png"
        main_save_path = crops_path / main_filename
        main_resized.save(main_save_path)
        
        # 2. Minimap Crop
        minimap_cropped = crop_minimap(img)
        minimap_filename = f"minimap_{idx}.png"
        minimap_save_path = crops_path / minimap_filename
        minimap_cropped.save(minimap_save_path)
        
        # 3. Label specification representation (mocked for initial pipeline test)
        # possession: 0 = User, 1 = Opponent, 2 = Loose
        # zone: 0 = Defensive, 1 = Middle, 2 = Attacking
        # ball: [x, y] coordinates
        possession = random.choice([0, 1, 2])
        zone = random.choice([0, 1, 2])
        ball = [random.uniform(0.1, 0.9), random.uniform(0.1, 0.9)]
        
        records.append({
            "original_path": str(filepath.resolve()),
            "main_image_path": str(main_save_path.resolve()),
            "minimap_image_path": str(minimap_save_path.resolve()),
            "possession": possession,
            "zone": zone,
            "ball": ball
        })
        print(f"Successfully processed: {filepath.name}")
    except Exception as e:
        print(f"Error processing {filepath.name}: {e}")

# Output matched record definitions
json_path = output_dir / "matched_records.json"
with open(json_path, "w") as f:
    json.dump(records, f, indent=2)
    
print(f"Success! Wrote matched records mapping to {json_path.resolve()}")